# Every number in the paper, from the pipeline outputs

This notebook **does not compute anything**. It reads the files written by `bash scripts/reproduzir.sh` and shows each
number of the paper next to where it appears and the file it comes from. Run the pipeline first; then run all cells.

Each table has three columns: **Paper** (where the number appears), **Value** (formatted as in the paper) and **Source**.
If `paper/main.tex` is present, the last cell checks that every value appears in the paper text.

In [1]:
import json, os, re
from decimal import Decimal, ROUND_HALF_UP
import pandas as pd
from IPython.display import display

ROOT = os.path.abspath(".")
TAB, VAL, NUM = f"{ROOT}/data/derived/tables", f"{ROOT}/data/validation", f"{ROOT}/data/derived/numeros"
csv = lambda p: pd.read_csv(p)
js = lambda p: json.load(open(p))
KN = js(f"{ROOT}/docs/key_numbers_final.json")
N_KEY, N_VS, N_SR, N_LIG = (js(f"{NUM}/{n}.json") for n in
    ["key_numbers", "validation_score", "spec_rot_viabilidade", "p4_ligacao"])

def pct(x, casas=1):  # proportion -> percentage, rounded half up from the 6-decimal output
    return str((Decimal(str(x)) * 100).quantize(Decimal(1).scaleb(-casas), rounding=ROUND_HALF_UP))
def dec(x, casas=2):
    return str(Decimal(str(x)).quantize(Decimal(1).scaleb(-casas), rounding=ROUND_HALF_UP))
def ci(lo, hi, sep=", "):
    return f"[{pct(lo)}{sep}{pct(hi)}]"
def n(x):
    return f"{int(x):,}"
def w(k):  # small counts are written out in the paper
    return ["zero", "one", "two", "three", "four", "five", "six", "seven", "eight", "nine", "ten"][int(k)]

CHECK = []  # (value, text expected in the paper)
def show(rows):
    for paper, value, source, *expect in rows:
        CHECK.append((paper, expect[0] if expect else value))
    display(pd.DataFrame([r[:3] for r in rows], columns=["Paper", "Value", "Source"], index=range(1, len(rows) + 1)))

## Data and units (Section III-A)

In [2]:
f = csv(f"{TAB}/funil.csv").set_index("etapa")
todas, sem_caf, ciclo = f.loc["todas as specs"], f.loc["sem caffeine"], f.loc["u_ciclo: ... e sem repos vendor"]
show([
    ("Spec files in SpecMine v1.0", n(todas.specs), "tables/funil.csv", f"{n(todas.specs)} spec files"),
    ("Repositories in SpecMine v1.0", n(todas.repos), "tables/funil.csv", f"{n(todas.repos)} repositories"),
    ("caffeine.ai specs excluded", n(todas.specs - sem_caf.specs), "tables/funil.csv", f"exclude {n(todas.specs - sem_caf.specs)}"),
    ("... across repositories", n(todas.repos - sem_caf.repos), "tables/funil.csv", f"across {n(todas.repos - sem_caf.repos)} repositories"),
    ("Deduplicated corpus: specs", n(ciclo.specs), "tables/funil.csv", f"{n(ciclo.specs)} specs"),
    ("Deduplicated corpus: repositories", n(ciclo.repos), "tables/funil.csv", f"{n(ciclo.repos)} repositories"),
])

,Paper,Value,Source
1,Spec files in SpecMine v1.0,"470,795",tables/funil.csv
2,Repositories in SpecMine v1.0,"73,031",tables/funil.csv
3,caffeine.ai specs excluded,"14,262",tables/funil.csv
4,... across repositories,"13,749",tables/funil.csv
5,Deduplicated corpus: specs,"447,795",tables/funil.csv
6,Deduplicated corpus: repositories,"57,435",tables/funil.csv


## Table I: RQ1 (edited; edited within the first 60 days)

In [3]:
c = csv(f"{TAB}/ciclo.csv").set_index("grupo")
j = csv(f"{TAB}/ciclo_janela60.csv").set_index("grupo")
h = csv(f"{TAB}/hipoteses.csv")
rows = []
for nome, g in [("OpenSpec living", "openspec:living"), ("Spec Kit", "speckit"), ("Other tools", "outras_nomeadas"), ("No tool", "sem_ferramenta")]:
    v = f"{pct(c.loc[g].por_repo)} {ci(c.loc[g].ic_repo_lo, c.loc[g].ic_repo_hi)}"
    rows.append((f"{nome}: edited [CI]; within 60 days", f"{v}; {pct(j.loc[g].por_repo)}", "tables/ciclo.csv, ciclo_janela60.csv",
                 f"{nome} & {v} & {pct(j.loc[g].por_repo)}"))
show(rows)

,Paper,Value,Source
1,OpenSpec living: edited [CI]; within 60 days,"28.4 [27.5, 29.2]; 28.9","tables/ciclo.csv, ciclo_janela60.csv"
2,Spec Kit: edited [CI]; within 60 days,"22.9 [22.2, 23.5]; 22.7","tables/ciclo.csv, ciclo_janela60.csv"
3,Other tools: edited [CI]; within 60 days,"17.8 [16.5, 19.1]; 17.2","tables/ciclo.csv, ciclo_janela60.csv"
4,No tool: edited [CI]; within 60 days,"40.0 [39.6, 40.5]; 38.1","tables/ciclo.csv, ciclo_janela60.csv"


## RQ1 in the text (Sections III-B and IV-A)

In [4]:
h1b = h[h.hipotese == "H1"].iloc[0]
idade = N_KEY["idade_mediana_dias"]
show([
    ("H1: difference, living vs Spec Kit (pp)", pct(h1b.estimativa), "tables/hipoteses.csv", f"{pct(h1b.estimativa)} percentage points"),
    ("H1: one-sided Mann-Whitney p", "p < 0.001" if h1b.mw_p_unilateral < 0.001 else str(h1b.mw_p_unilateral), "tables/hipoteses.csv", "p < 0.001"),
    ("H1: Cliff's delta", dec(h1b.cliff_delta), "tables/hipoteses.csv", f"delta = {dec(h1b.cliff_delta)}"),
    ("Median age, Spec Kit (days)", f"{idade['speckit']:.0f}", "numeros/key_numbers.json", f"median {idade['speckit']:.0f} days"),
    ("Median age, living specs (days)", f"{idade['openspec:living']:.0f}", "numeros/key_numbers.json", f"versus {idade['openspec:living']:.0f} for living"),
    ("60-day window: living vs Spec Kit", f"{pct(j.loc['openspec:living'].por_repo)}% vs. {pct(j.loc['speckit'].por_repo)}%",
     "tables/ciclo_janela60.csv", f"{pct(j.loc['openspec:living'].por_repo)}% vs. {pct(j.loc['speckit'].por_repo)}%"),
])

,Paper,Value,Source
1,"H1: difference, living vs Spec Kit (pp)",5.5,tables/hipoteses.csv
2,H1: one-sided Mann-Whitney p,p < 0.001,tables/hipoteses.csv
3,H1: Cliff's delta,0.17,tables/hipoteses.csv
4,"Median age, Spec Kit (days)",101,numeros/key_numbers.json
5,"Median age, living specs (days)",69,numeros/key_numbers.json
6,60-day window: living vs Spec Kit,28.9% vs. 22.7%,tables/ciclo_janela60.csv


## RQ2 (spec decay): sample and method (Section III-C)

In [5]:
pw = csv(f"{TAB}/p4_poder.csv")
esc = pw[pw.principal.astype(str) == "True"]
show([
    ("Eligible repositories / candidate pairs", f"{N_SR['candidatos_repos']} / {n(N_SR['candidatos'])}", "numeros/spec_rot_viabilidade.json",
     f"The {N_SR['candidatos_repos']} eligible repositories contained {n(N_SR['candidatos'])} candidate pairs"),
    ("Pairs selected (up to 3 per repository)", str(N_SR["pares_sorteados"]), "numeros/spec_rot_viabilidade.json", f"({N_SR['pares_sorteados']} selected)"),
    ("Power simulation: pilot decay rate", pct(esc.y.iloc[0]) + "%", "tables/p4_poder.csv", f"assuming a {pct(esc.y.iloc[0])}% decay rate"),
    ("Pairs with diffs / judgeable", f"{N_VS['pares']} / {N_VS['julgaveis']}", "numeros/validation_score.json",
     f"{N_VS['julgaveis']} were judgeable"),
    ("Distinct PRs / specs among judgeable", f"{N_VS['prs_unicos']} / {N_VS['specs_unicas']}", "numeros/validation_score.json",
     f"({N_VS['prs_unicos']} distinct PRs, {N_VS['specs_unicas']} specs)"),
    ("Excluded: code already contradicted the spec", str(N_VS["divergencia_previa"]), "numeros/validation_score.json",
     f"({N_VS['divergencia_previa']} pairs)"),
    ("Owner check: other capability / confirmed / no evidence / vetoed",
     f"{N_VS['dona_outra_capacidade']} / {N_VS['dona_confirmadas']} / {N_VS['dona_sem_evidencia']} / {N_VS['vetos_iniciais']}",
     "numeros/validation_score.json",
     f"Of {N_VS['dona_outra_capacidade']} such labels, {N_VS['dona_confirmadas']} were confirmed, {N_VS['dona_sem_evidencia']} lacked evidence, and the rater vetoed {N_VS['vetos_iniciais']}"),
    ("Vetoes withdrawn", str(N_VS["vetos_iniciais"] - N_VS["vetos_finais"]), "numeros/validation_score.json", f"withdrew {w(N_VS['vetos_iniciais'] - N_VS['vetos_finais'])}"),
    ("Agreement measured on the pairs outside the pilot", str(N_VS["kappa_n"]), "numeros/validation_score.json",
     f"agreement is measured on the {N_VS['kappa_n']} pairs outside the pilot"),
    ("Pairs adjudicated (human vs DeepSeek disagree on decay)", str(N_VS["adjudicados"]), "numeros/validation_score.json",
     f"On the {N_VS['adjudicados']} pairs"),
])

,Paper,Value,Source
1,Eligible repositories / candidate pairs,"108 / 13,028",numeros/spec_rot_viabilidade.json
2,Pairs selected (up to 3 per repository),289,numeros/spec_rot_viabilidade.json
3,Power simulation: pilot decay rate,12.5%,tables/p4_poder.csv
4,Pairs with diffs / judgeable,280 / 270,numeros/validation_score.json
5,Distinct PRs / specs among judgeable,198 / 239,numeros/validation_score.json
6,Excluded: code already contradicted the spec,3,numeros/validation_score.json
7,Owner check: other capability / confirmed / no...,128 / 115 / 13 / 12,numeros/validation_score.json
8,Vetoes withdrawn,1,numeros/validation_score.json
9,Agreement measured on the pairs outside the pilot,222,numeros/validation_score.json
10,Pairs adjudicated (human vs DeepSeek disagree ...,28,numeros/validation_score.json


## Table II: RQ2 spec decay Y (% of judgeable pairs, mean per repository, 95% CI)

In [6]:
r = csv(f"{VAL}/resultado_p4.csv").set_index(["amostra", "medida"])
linhas = [("Main", "principal"), ("Excluding pilot pairs", "(a) só os 222 pares fora do piloto"), ("Before adjudication", "antes da adjudicação"),
          ("Component-level owner = decay", "(h) dona só por componente = omite"), ("Spec. gaps = decay", "final sem o veto humano"),
          ("Gaps and component = decay", "final sem o veto e dona só por componente = omite"),
          ("Cannot judge = decay", "(d2) não dá para julgar = decay"),
          ("PRs >=60 days old", "(e) só PRs >= 60 dias antes do corte")]
rows = []
for nome, k in linhas:
    b, s = r.loc[(k, "amplo (critério)")], r.loc[(k, "estrito")]
    v = f"{b.pares} & {pct(b.y)} {ci(b.ic_lo, b.ic_hi)} & {pct(s.y)} {ci(s.ic_lo, s.ic_hi)}"
    rows.append((nome, f"pairs {b.pares}; broad {pct(b.y)} {ci(b.ic_lo, b.ic_hi)}; strict {pct(s.y)} {ci(s.ic_lo, s.ic_hi)}; {b.leitura}",
                 "validation/resultado_p4.csv", v))
show(rows)

,Paper,Value,Source
1,Main,"pairs 270; broad 10.6 [5.9, 16.0]; strict 5.4 ...",validation/resultado_p4.csv
2,Excluding pilot pairs,"pairs 213; broad 10.2 [5.7, 15.2]; strict 4.4 ...",validation/resultado_p4.csv
3,Before adjudication,"pairs 270; broad 7.3 [4.1, 10.8]; strict 2.7 [...",validation/resultado_p4.csv
4,Component-level owner = decay,"pairs 270; broad 13.5 [8.7, 18.9]; strict 5.4 ...",validation/resultado_p4.csv
5,Spec. gaps = decay,"pairs 270; broad 14.9 [9.5, 20.8]; strict 5.4 ...",validation/resultado_p4.csv
6,Gaps and component = decay,"pairs 270; broad 17.8 [12.4, 24.0]; strict 5.4...",validation/resultado_p4.csv
7,Cannot judge = decay,"pairs 277; broad 12.4 [7.6, 18.1]; strict 7.3 ...",validation/resultado_p4.csv
8,PRs >=60 days old,"pairs 119; broad 12.3 [4.7, 20.1]; strict 6.3 ...",validation/resultado_p4.csv


## RQ2 in the text (Section IV-B)

In [7]:
xa = N_LIG["x_amplo"]
rb = N_VS["rotulo_bruto"]
oc = N_VS["outra_capacidade_final"]
cz = csv(f"{VAL}/resultado_p4_confusao.csv").set_index("humano_bruto")
pb, ps = r.loc[("principal", "amplo (critério)")], r.loc[("principal", "estrito")]
semdona = r.loc[("(c) sem a revisão de dona", "amplo (critério)")]
show([
    ("X: linked living specs with such a PR [CI]", f"{pct(N_SR['x_por_repo'])}% {ci(*N_SR['x_ic'])}", "numeros/spec_rot_viabilidade.json",
     f"{pct(N_SR['x_por_repo'])}% {ci(*N_SR['x_ic'])}"),
    ("X: repositories", str(N_SR["x_repos"]), "numeros/spec_rot_viabilidade.json", f"({N_SR['x_repos']} repositories)"),
    ("Human labels: preserves / other capability / omits / contradicts / cannot judge",
     f"{rb['preserva']} / {rb['outra_capacidade']} / {rb['omite']} / {rb['contradiz']} / {rb['nao_da_para_julgar']}",
     "numeros/validation_score.json",
     f"{rb['preserva']} preserves, {rb['outra_capacidade']} other capability, {rb['omite']} omits, {rb['contradiz']} contradicts, and {rb['nao_da_para_julgar']} cannot judge"),
    ("Strict Y [CI]; pooled", f"{pct(ps.y)}% {ci(ps.ic_lo, ps.ic_hi)}; {ps.positivos} of {ps.pares}", "validation/resultado_p4.csv",
     f"{ps.positivos} of {ps.pares} pooled"),
    ("Broad Y [CI]; pooled", f"{pct(pb.y)}% {ci(pb.ic_lo, pb.ic_hi)}; {pb.positivos} of {pb.pares}", "validation/resultado_p4.csv",
     f"{pb.positivos} of {pb.pares} pooled"),
    ("Pairs ending as other capability: total / described / component / gaps",
     f"{sum(oc.values())} / {oc['explicita']} / {oc['componente']} / {oc['veto']}", "numeros/validation_score.json",
     f"In {sum(oc.values())} of the {N_VS['pares']} pairs"),
    ("Intra-repository correlation: broad / strict", f"{dec(N_VS['icc_amplo'])} / {dec(N_VS['icc_estrito'])}", "numeros/validation_score.json",
     f"({dec(N_VS['icc_amplo'])} broad, {dec(N_VS['icc_estrito'])} strict)"),
    ("kappa human vs DeepSeek: five labels", dec(N_VS["kappa_5_categorias"]), "numeros/validation_score.json",
     f"kappa = {dec(N_VS['kappa_5_categorias'])}"),
    ("kappa: decay yes/no", dec(N_VS["kappa_binario"]), "numeros/validation_score.json", f"and {dec(N_VS['kappa_binario'])} for decay"),
    ("kappa: pairs (five labels / decay yes-no)", f"{N_VS['kappa_n']} / {N_VS['kappa_binario_n']}", "numeros/validation_score.json",
     f"labels ({N_VS['kappa_n']} pairs) and {dec(N_VS['kappa_binario'])} for decay yes/no ({N_VS['kappa_binario_n']} judgeable pairs)"),
    ("Decay counts: human / DeepSeek, of blind pairs", f"{N_VS['decay_humano']} / {N_VS['decay_deepseek']} of {N_VS['kappa_binario_n']}",
     "numeros/validation_score.json", f"({N_VS['decay_humano']} and {N_VS['decay_deepseek']} of {N_VS['kappa_binario_n']})"),
    ("Human preserves, LLM omits/contradicts (blind)", str(cz.loc["preserva", "omite"] + cz.loc["preserva", "contradiz"]),
     "validation/resultado_p4_confusao.csv", f"In {cz.loc['preserva', 'omite'] + cz.loc['preserva', 'contradiz']} pairs the human saw"),
    ("Human omits, LLM other capability (blind)", str(cz.loc["omite", "outra_capacidade"]), "validation/resultado_p4_confusao.csv",
     f"In {cz.loc['omite', 'outra_capacidade']} pairs the human labeled"),
    ("Adjudication moved into / out of decay", f"{N_VS['adjudicacao_entram']} / {N_VS['adjudicacao_saem']}", "numeros/validation_score.json",
     f"moved {N_VS['adjudicacao_entram']} pairs into decay and {N_VS['adjudicacao_saem']} out"),
])

,Paper,Value,Source
1,X: linked living specs with such a PR [CI],"28.3% [24.0, 32.7]",numeros/spec_rot_viabilidade.json
2,X: repositories,190,numeros/spec_rot_viabilidade.json
3,Human labels: preserves / other capability / o...,125 / 128 / 11 / 9 / 7,numeros/validation_score.json
4,Strict Y [CI]; pooled,"5.4% [2.2, 9.5]; 12 of 270",validation/resultado_p4.csv
5,Broad Y [CI]; pooled,"10.6% [5.9, 16.0]; 26 of 270",validation/resultado_p4.csv
6,Pairs ending as other capability: total / desc...,131 / 111 / 9 / 11,numeros/validation_score.json
7,Intra-repository correlation: broad / strict,0.32 / 0.43,numeros/validation_score.json
8,kappa human vs DeepSeek: five labels,0.33,numeros/validation_score.json
9,kappa: decay yes/no,0.23,numeros/validation_score.json
10,kappa: pairs (five labels / decay yes-no),222 / 213,numeros/validation_score.json


## Threats to validity (Section VI)

In [8]:
show([
    ("tasks.md link supported by SpecMine", f"{N_VS['ligacao_confirmada']} of {N_VS['pares']}", "numeros/validation_score.json",
     f"{N_VS['ligacao_confirmada']} of {N_VS['pares']} pairs"),
    ("Y without the owner check", pct(semdona.y) + "%", "validation/resultado_p4.csv", f"lowers Y ({pct(semdona.y)}%)"),
    ("Pairs whose spec commit was committed after the PR merge / counted as decay",
     f"{N_VS['commit_depois_do_merge']} / {N_VS['commit_depois_do_merge_decay']}", "numeros/validation_score.json",
     f"In {N_VS['commit_depois_do_merge']} of the {N_VS['pares']} pairs"),
])

,Paper,Value,Source
1,tasks.md link supported by SpecMine,279 of 280,numeros/validation_score.json
2,Y without the owner check,6.7%,validation/resultado_p4.csv
3,Pairs whose spec commit was committed after th...,2 / 1,numeros/validation_score.json


## Every other number in the paper

Method parameters are read from the scripts that use them, verdict words are derived from the output tables, and
results repeated in the abstract, discussion, and conclusion are pinned in their own sentence. Numbers taken from
cited work are listed with the passage of the source that states them; that passage is checked when the source text
is available (`literatura/`, kept in the project but not shipped, for copyright).

In [9]:
src = lambda p: open(f"{ROOT}/{p}", encoding="utf-8").read()
def num(pat, p):
    m = re.search(pat, src(p))
    assert m, (pat, p)
    return m.group(1)
cmc = f.loc["sem caffeine, com commit até o corte"]
CO = N_KEY["corpus"]
corte = pd.Timestamp(KN["corte"])
dia = f"{corte:%B} {corte.day}"  # the paper names the end of SpecMine's history collection by its date
kn_src, vs_src = src("scripts/analysis/key_numbers.py"), src("scripts/analysis/validation_score.py")
ic95 = all(f"np.percentile({v}, {q})" in s for s, v in [(kn_src, "bm"), (vs_src, "bm")] for q in ("2.5", "97.5"))
janela = int(num(r"interval (\d+) day", "scripts/analysis/key_numbers.py"))
d60 = int(num(r"dias_ao_corte >= (\d+)", "scripts/analysis/validation_score.py"))
copia = int(num(r"COPIA_MIN_REPOS = (\d+)", "scripts/analysis/universe.py"))
vendor = float(num(r"VENDOR_FRAC = ([\d.]+)", "scripts/analysis/universe.py"))
m_repo = int(num(r"M_POR_REPO = (\d+)", "scripts/analysis/spec_rot_viabilidade.py"))
genericos = [re.sub(r"(\w+)s\?$", r"\1(s)", g) for g in
             num(r"\(\^\|/\)\(([a-z|?]+)\)", "scripts/analysis/spec_rot_viabilidade.py").split("|")]
lim, alvo = map(float, re.search(r"LIMIAR, ALVO = \d+, \d+, ([\d.]+), ([\d.]+)", src("scripts/analysis/p4_poder.py")).groups())
b_neg, b_dec = map(float, re.search(r'"resultado negativo" if hi < ([\d.]+) else "decay" if lo >= ([\d.]+)', vs_src).groups())
poder_ok = esc.groupby("m").poder.min()
m_rec, rho_max = int(poder_ok[poder_ok >= alvo].index.min()), float(esc.rho.max())
h1a = h[h.hipotese == "menos_da_metade"]
neg = lambda x: x.leitura == "resultado negativo"
gap = r.loc[("final sem o veto humano", "amplo (critério)")]
ant = r.loc[("(e) só PRs >= 60 dias antes do corte", "amplo (critério)")]
fora_piloto = r.loc[("(a) só os 222 pares fora do piloto", "amplo (critério)")]
cruza = {nome for nome, k in linhas if not neg(r.loc[(k, "amplo (critério)")])}
estrito_neg = all(neg(r.loc[(k, "estrito")]) for _, k in linhas)
folga = pct(ant.ic_hi - lim)
pct0 = lambda x: pct(x, 0)
se = lambda cond, txt: txt if cond else "MISMATCH: " + txt
n_llm = len(csv(f"{VAL}/llm/llm_pares_v3.csv"))
n_hum = sum(rb.values())
reg = src("data/validation/llm/MODELO_E_PROMPTS.md")
oc_e, oc_c, oc_v = oc["explicita"], oc["componente"], oc["veto"]
rows = [
    ("Method: files whose commit history SpecMine did not collect (= funnel step 'commit by the cutoff')",
     n(CO["historico_nao_coletado"]), "numeros/key_numbers.json; tables/funil.csv",
     se(CO["historico_nao_coletado"] == sem_caf.specs - cmc.specs,
        f"files with no commit history in SpecMine ({n(CO['historico_nao_coletado'])}), since their edits cannot be counted")),
    ("Method: collection dates (history; contents); RQ2 specs changed between them", f"{CO['coleta_historico']}; {CO['coleta_texto']}; {N_VS['texto_mudou_entre_coletas']} of {N_VS['texto_pares_conferidos']}",
     "numeros/key_numbers.json; numeros/validation_score.json",
     se(len(set(CO["coleta_historico"])) == 1 and len(set(CO["coleta_texto"])) == 1 and CO["coleta_historico"][0] == f"{corte:%Y-%m-%d}"
        and N_VS["texto_mudou_entre_coletas"] == 0 and corte.strftime("%H:%M:%S") == "23:59:59",
        f"SpecMine fetched commit histories on {dia}, {corte.year} and spec contents on {pd.Timestamp(CO['coleta_texto'][0]):%B} {pd.Timestamp(CO['coleta_texto'][0]).day}; "
        f"we ignore commits and PRs dated after {dia} (UTC) and use each spec's text as fetched (none of the {N_VS['texto_pares_conferidos']} RQ2 specs changed between the two dates)")),
    ("Method: copied content / vendor threshold", f"{copia} repositories / {vendor}", "scripts/analysis/universe.py",
     se(vendor == 0.5, f"appears in at least {w(copia)} repositories; and repositories where at least half the specs")),
    ("Method: SpecMine version name", f"{corte:%Y-%m}", "docs/key_numbers_final.json", f"v1.0 ({corte:%B %Y} snapshot)"),
    ("RQ1: edited = second commit by the end of the history collection", dia, "docs/key_numbers_final.json", f"has a second commit by {dia}, regardless of age"),
    ("Method: bootstrap resamples, 95% percentile interval", f"{KN['replicas_bootstrap']}; 2.5/97.5 percentiles", "key_numbers_final.json; scripts",
     se(ic95, f"with 95% confidence intervals from {n(KN['replicas_bootstrap'])} bootstrap resamples of repositories")),
    ("H1 criterion", "nonoverlapping 95% CIs", "scripts/analysis/key_numbers.py", se(ic95, "requires both nonoverlapping 95% intervals")),
    ("RQ1 window (days)", str(janela), "scripts/analysis/key_numbers.py",
     f"a common {janela}-day window, shorter than both median ages: among specs at least {janela} days old on {dia}, we record whether a second commit occurred within {janela} days of creation"),
    ("RQ2 window end", dia, "docs/key_numbers_final.json", f"merged after its last edit but by {dia}. The PR must"),
    ("RQ2: text at PR merge", dia, "docs/key_numbers_final.json", f"With no spec edit between the PR and {dia}, the fetched text is also the text at PR merge"),
    ("RQ1 results: by the end of the history collection", dia, "docs/key_numbers_final.json", f"By {dia}, fewer than half"),
    ("Threats: nothing after the history collection", dia, "docs/key_numbers_final.json", f"Updates after {dia} are unobserved"),
    ("RQ2 pairs per repository", str(m_repo), "scripts/analysis/spec_rot_viabilidade.py", f"up to {w(m_repo)} spec--PR pairs per repository"),
    ("Power rule: rho, cutoff, power; recomputed m", f"{rho_max} / {lim} / {alvo}; m = {m_rec}", "tables/p4_poder.csv; p4_poder.py",
     se(m_rec == m_repo, f"<= {rho_max}, the upper confidence limit fell below our {pct0(lim)}% cutoff in at least {pct0(alvo)}% of simulated samples")),
    ("Pairs with diffs, all read", str(N_VS["pares"]), "numeros/validation_score.json",
     se(n_hum == N_VS["pares"], f"returned diffs for {N_VS['pares']} pairs; one author read all {n_hum}")),
    ("Timestamp exceptions", str(N_VS["commit_depois_do_merge"]), "numeros/validation_score.json",
     f"{w(N_VS['commit_depois_do_merge'])} timestamp exceptions"),
    ("Owner by component only", str(oc_c), "numeros/validation_score.json", f"({oc_c} pairs; Table"),
    ("DeepSeek labels", str(n_llm), "validation/llm/llm_pares_v3.csv", f"applied the same codebook to all {n_llm} pairs"),
    ("H2 bands", f"{b_neg} / {b_dec}", "scripts/analysis/validation_score.py",
     se(lim == b_neg, f"entirely below {pct0(b_neg)}% is a negative result; within {pct0(b_neg)}--{pct0(b_dec)}%, co-change with some decay; at least {pct0(b_dec)}%, decay")),
    ("Figure 1: end of the history", dia, "docs/key_numbers_final.json", f"{dia}, {corte.year}: end of the commit history"),
    ("Footnote: excluded generic basenames", ", ".join(genericos), "scripts/analysis/spec_rot_viabilidade.py",
     "Excluded basenames, with any extension: " + ", ".join(genericos) + "."),
    ("Table II caption", f"strict always negative: {estrito_neg}; broad crosses in {sorted(cruza)}", "validation/resultado_p4.csv",
     se(estrito_neg and cruza == {"Spec. gaps = decay", "Gaps and component = decay", "PRs >=60 days old"},
        f"Strict Y stays below {pct0(lim)}% everywhere; broad Y crosses it when specification gaps count as decay and, by {folga} points, for older PRs.")),
    ("Fewer than half edited (all groups)", ", ".join(h1a.veredito), "tables/hipoteses.csv",
     se((h1a.veredito == "satisfeito").all() and h1a.limiar.iloc[0] == 0.5, "fewer than half of the specs in each group had been edited")),
    ("H1 verdict and effect size", f"{h1b.veredito}; {h1b.magnitude}", "tables/hipoteses.csv",
     se(h1b.veredito == "satisfeito" and h1b.magnitude == "small", "indicates a small effect (H1 met)")),
    ("60-day window keeps the order", f"{pct(j.loc['openspec:living'].por_repo)} > {pct(j.loc['speckit'].por_repo)}", "tables/ciclo_janela60.csv",
     se(j.loc["openspec:living"].por_repo > j.loc["speckit"].por_repo, f"The order also holds in the fixed {janela}-day window")),
    ("Human labels total", str(n_hum), "numeros/validation_score.json", f"In the {n_hum} pairs, the human labels were"),
    ("Strict Y negative in every analysis of Table II", str(estrito_neg), "validation/resultado_p4.csv",
     se(estrito_neg, "a negative result in every analysis of Table")),
    ("Broad Y negative: main and outside the pilot", f"{pb.leitura}; {fora_piloto.leitura}", "validation/resultado_p4.csv",
     se(neg(pb) and neg(fora_piloto), "also a negative result under the main rule, including without the pilot pairs")),
    ("Older PRs only", f"{ant.pares}; {pct(ant.y)} {ci(ant.ic_lo, ant.ic_hi)}; {ant.leitura}", "validation/resultado_p4.csv",
     se(ant.leitura == "inconclusivo" and d60 == janela, f"we restrict RQ2 to {ant.pares} pairs with no spec update for at least {d60} days between PR merge and {dia}; Y = {pct(ant.y)}% {ci(ant.ic_lo, ant.ic_hi)}, inconclusive by {folga} points")),
    ("Specification gaps", str(oc_v), "numeros/validation_score.json",
     se(N_VS["vetos_finais"] == oc_v, f"In {oc_v} pairs, the rater placed the changed behavior outside the spec's scope")),
    ("... veto kept on all gaps", str(N_VS["vetos_finais"]), "numeros/validation_score.json", f"kept the veto on all {N_VS['vetos_finais']} pairs"),
    ("Gaps = decay", f"{pct(gap.y)} {ci(gap.ic_lo, gap.ic_hi)}; {gap.leitura}", "validation/resultado_p4.csv",
     se(gap.leitura == "inconclusivo", f"raises Y from {pct(pb.y)}% to {pct(gap.y)}% {ci(gap.ic_lo, gap.ic_hi)}, making RQ2 inconclusive")),
    ("Other capability: described / component / gaps", f"{oc_e} / {oc_c} / {oc_v}", "numeros/validation_score.json",
     f"In {oc_e} of them another spec of the repository describes the changed behavior, in {oc_c} another spec covers only the component, and {oc_v} are the specification gaps"),
    ("ICC above the assumed rho; verdict unaffected", f"{dec(N_VS['icc_amplo'])} > {rho_max}; CI hi {pct(pb.ic_hi)}% < {pct0(lim)}%",
     "numeros/validation_score.json", se(N_VS["icc_amplo"] > rho_max and N_VS["icc_estrito"] > rho_max and pb.ic_hi < lim,
        f"exceeds the \\rho <= {rho_max} assumed when sizing the sample; since the interval already excludes the threshold")),
    ("Discussion: described elsewhere", str(oc_e), "numeros/validation_score.json", f"In {oc_e} sampled pairs, another spec"),
    ("Discussion: the gaps decide the verdict", f"{pb.leitura} vs {gap.leitura}", "validation/resultado_p4.csv",
     se(neg(pb) and not neg(gap), f"The {oc_v} gaps with no known owner decide whether broad decay falls below {pct0(lim)}%")),
    ("Threats: gaps", str(oc_v), "numeros/validation_score.json", f"The {oc_v} specification gaps rest"),
    ("Threats: late commit counted as decay", str(N_VS["commit_depois_do_merge_decay"]), "numeros/validation_score.json",
     f"{w(N_VS['commit_depois_do_merge_decay'])} of them counts as decay"),
    ("LLM registry: DeepSeek model, temperature, date", "deepseek-v4-pro; 0,3; nov/2026", "validation/llm/MODELO_E_PROMPTS.md",
     se("deepseek-v4-pro" in reg and "temperatura 0,3" in reg and "nov/2026" in reg, "DeepSeek v4 Pro (temperature 0.3, November 2026)")),
    ("LLM registry: Claude model", "Claude Opus 5.5", "validation/llm/MODELO_E_PROMPTS.md",
     se("Claude Opus 5.5" in reg, "Claude Opus 5.5 reviewed the owner evidence")),
    ("Abstract: pairs read", str(N_VS["pares"]), "numeros/validation_score.json", f"read {N_VS['pares']} pairs of an OpenSpec"),
    ("Abstract: most specs never re-edited", "all groups < 50%", "tables/ciclo.csv",
     se((c.por_repo < 0.5).all(), f"Most specs had no later commit by {dia}, {corte.year}")),
    ("Abstract: small difference", h1b.magnitude, "tables/hipoteses.csv", se(h1b.magnitude == "small", "although the difference was small")),
    ("Abstract: Y strict / broad", f"{pct(ps.y)} / {pct(pb.y)}", "validation/resultado_p4.csv",
     f"Of the {pb.pares} judgeable pairs, the mean per-repository share whose spec became false was {pct(ps.y)}%; counting specs that also failed to describe new behavior within their scope raised it to {pct(pb.y)}%"),
    ("Abstract: gaps", f"{oc_v}; {pct(gap.y)} {ci(gap.ic_lo, gap.ic_hi)}", "validation/resultado_p4.csv",
     f"In {oc_v} pairs, the changed behavior was outside the focal spec's scope, and our search found no other spec describing it. Treating these gaps as decay raised the estimate to {pct(gap.y)}% (95% CI {pct(gap.ic_lo)}--{pct(gap.ic_hi)})"),
    ("Introduction: pairs read", str(N_VS["pares"]), "numeros/validation_score.json", f"read {N_VS['pares']} spec--PR pairs by hand"),
    ("Conclusion: Y and bands", f"{pct(ps.y)} / {pct(pb.y)}", "validation/resultado_p4.csv",
     se(ps.ic_hi < lim and pb.ic_hi < lim, f"false in {pct(ps.y)}% of pairs and false or incomplete in {pct(pb.y)}% (repository means), both below {pct0(lim)}%")),
    ("Conclusion: inconclusive cases", f"{d60} days; {oc_v} gaps", "validation/resultado_p4.csv",
     se(ant.leitura == gap.leitura == "inconclusivo", f"no spec update for {d60} days or when {oc_v} specification gaps count as decay")),
    ("Conclusion: broad Y inconclusive cases", f"{ant.leitura}; {gap.leitura}", "validation/resultado_p4.csv",
     se(ant.leitura == gap.leitura == "inconclusivo", "The latter becomes inconclusive for pairs")),
    ("RQ1 heading: most specs have one commit", "all groups < 50%", "tables/ciclo.csv",
     se((c.por_repo < 0.5).all(), "Most Specs Have One Recorded Commit")),
    ("Table I and II captions: 95% CI", "2.5/97.5 percentiles", "scripts", se(ic95, "(per repository, 95% CI)")),
    ("... Table II", "2.5/97.5 percentiles", "scripts", se(ic95, "mean per repository, 95% CI)")),
    ("Table II row: older PRs", f">= {d60} days", "scripts/analysis/validation_score.py", f"PRs >={d60} days old"),
    ("RQ2 text: strict Y [CI]", f"{pct(ps.y)}% {ci(ps.ic_lo, ps.ic_hi)}", "validation/resultado_p4.csv",
     f"became false in {pct(ps.y)}% {ci(ps.ic_lo, ps.ic_hi)} of the pairs"),
    ("RQ2 text: broad Y [CI]", f"{pct(pb.y)}% {ci(pb.ic_lo, pb.ic_hi)}", "validation/resultado_p4.csv",
     f"Y = {pct(pb.y)}% {ci(pb.ic_lo, pb.ic_hi)} (broad Y;"),
    ("kappa over the label categories", str(len(rb)), "numeros/validation_score.json", f"over the {w(len(rb))} labels"),
    ("Agreement called weak (kappa < 0.4, docs/METRICAS.md)", f"{dec(N_VS['kappa_5_categorias'])}, {dec(N_VS['kappa_binario'])}", "numeros/validation_score.json",
     se(max(N_VS["kappa_5_categorias"], N_VS["kappa_binario"]) < 0.4, "outside the pilot was weak")),
    ("RQ2 heading: strict Y negative everywhere", str(estrito_neg), "validation/resultado_p4.csv", se(estrito_neg, "Living Specs Rarely Become False")),
    ("Corpus: years of first commit; repositories created in 2025 or later", f"{'--'.join(CO['primeiro_commit_anos'])}; {n(CO['repos_criados_2025_ou_depois'])} of {n(CO['repos'])}",
     "numeros/key_numbers.json", se(CO["repos_criados_2025_ou_depois"] / CO["repos"] > 0.5,
                                    f"first committed in {'--'.join(CO['primeiro_commit_anos'])}, mostly in young repositories")),
    ("PR layer: fewest stars of a repository", str(CO["estrelas_min_camada_pr"]), "numeros/key_numbers.json",
     se(CO["estrelas_min_camada_pr"] >= 10, "repositories with at least 10 stars and to PRs that touch some spec")),
    ("Abstract: living edited more often", f"{h1b.veredito}; +{pct(h1b.estimativa)} pp", "tables/hipoteses.csv",
     se(h1b.veredito == "satisfeito" and h1b.estimativa > 0, "OpenSpec living specs were edited more often than Spec Kit specs")),
    ("Spec Kit specs older than living specs", f"{idade['speckit']:.0f} vs {idade['openspec:living']:.0f} days", "numeros/key_numbers.json",
     se(idade["speckit"] > idade["openspec:living"], "Spec Kit specs are older")),
    ("Window shorter than both median ages", f"{janela} < {min(idade['speckit'], idade['openspec:living']):.0f}", "numeros/key_numbers.json",
     se(janela < min(idade["speckit"], idade["openspec:living"]), "shorter than both median ages")),
    ("Living edited more often (percentage points)", pct(h1b.estimativa), "tables/hipoteses.csv",
     se(h1b.estimativa > 0, "percentage points more often than Spec Kit specs")),
    ("Table I caption: 60-day window", str(janela), "scripts/analysis/key_numbers.py",
     f"among specs at least {janela} days old, % edited within their first {janela} days"),
    ("Table I header: 60-day window", str(janela), "scripts/analysis/key_numbers.py", f"Edited & Within {janela} days"),
    ("Discussion: strict decay rare", str(estrito_neg), "validation/resultado_p4.csv",
     se(estrito_neg, "rarely left an OpenSpec living spec false")),
    ("Discussion: candidate PRs that touch no spec", f"{N_SR['candidatos_sem_spec']} of {n(N_SR['candidatos'])}", "numeros/spec_rot_viabilidade.json",
     se(N_SR["candidatos_sem_spec"] / N_SR["candidatos"] < 0.01, "so nearly every candidate PR updated a spec other than the focal one")),
    ("Discussion: living edited more often", h1b.veredito, "tables/hipoteses.csv",
     se(h1b.veredito == "satisfeito", "are edited more often than spec-first Spec Kit specs")),
    ("Discussion: small difference", h1b.magnitude, "tables/hipoteses.csv", se(h1b.magnitude == "small", "The difference is small")),
    ("Discussion: unedited living specs rarely become false", str(estrito_neg), "validation/resultado_p4.csv",
     se(estrito_neg, "it rarely becomes false")),
    ("Discussion: agreement weak", f"{dec(N_VS['kappa_5_categorias'])}, {dec(N_VS['kappa_binario'])}", "numeros/validation_score.json",
     se(max(N_VS["kappa_5_categorias"], N_VS["kappa_binario"]) < 0.4, "Agreement between the human and DeepSeek was weak")),
    ("Conclusion: small margin", f"{h1b.veredito}; {h1b.magnitude}", "tables/hipoteses.csv",
     se(h1b.veredito == "satisfeito" and h1b.magnitude == "small", "by a small margin")),
    ("Decay concentrates in some repositories", f"ICC {dec(N_VS['icc_amplo'])} / {dec(N_VS['icc_estrito'])}", "numeros/validation_score.json",
     se(N_VS["icc_amplo"] > 0 and N_VS["icc_estrito"] > 0, "Decay concentrates in some repositories")),
]
show(rows)

,Paper,Value,Source
1,Method: files whose commit history SpecMine di...,"1,473",numeros/key_numbers.json; tables/funil.csv
2,Method: collection dates (history; contents); ...,"['2026-07-11', '2026-07-11']; ['2026-07-12', '...",numeros/key_numbers.json; numeros/validation_s...
3,Method: copied content / vendor threshold,5 repositories / 0.5,scripts/analysis/universe.py
4,Method: SpecMine version name,2026-07,docs/key_numbers_final.json
5,RQ1: edited = second commit by the end of the ...,July 11,docs/key_numbers_final.json
...,...,...,...
70,Discussion: small difference,small,tables/hipoteses.csv
71,Discussion: unedited living specs rarely becom...,True,validation/resultado_p4.csv
72,Discussion: agreement weak,"0.33, 0.23",numeros/validation_score.json
73,Conclusion: small margin,satisfeito; small,tables/hipoteses.csv


### Numbers from cited work

In [10]:
ART = f"{ROOT}/literatura/artigos"
def texto(nome):
    caminho = f"{ART}/{nome}"
    if not os.path.exists(caminho):
        return None
    if nome.endswith(".pdf"):
        import shutil, subprocess
        if not shutil.which("pdftotext"):
            return None
        # also the -layout extraction, which keeps table rows (e.g., Appendix J of [1]) on one line
        return re.sub(r"\s+", " ", " ".join(subprocess.run(["pdftotext", *op, caminho, "-"], capture_output=True, text=True).stdout
                                            for op in ([], ["-layout"])))
    return re.sub(r"\s+", " ", open(caminho, encoding="utf-8", errors="ignore").read())
SM, EP, TA, AG = "2608.25202v5.pdf", "2610.07534.pdf", "tan2024_emse_outdated_code_refs.pdf", "aghajani2019_icse_documentation_issues.pdf"
citados = [  # (paper text, reference, source file, passage in the source)
    ("21 interviews, and 266 repositories", "[1] Agarwal et al.", SM, "interviews with 21 practitioners, including eight tool makers, and 266 active and popular repositories"),
    ("81.5% of specs had one commit", "[1] Agarwal et al.", SM, "81.5% of spec files have a single recorded commit"),
    ("7 interviewees raised spec decay", "[1] Agarwal et al., Appendix J", SM, "spec goes stale as code evolves P2, P8, P10, P11, P12, P14, P21"),
    ("(73,030 in", "[1] Agarwal et al.", SM, "470,795 spec files from 73,030 public GitHub repositories"),
    ("SpecMine collected PRs only from repositories with at least 10 stars", "[1] Agarwal et al.", SM,
     "we sweep every pull request that touches a spec in their repositories with at least ten stars"),
    ("repositories with at least 10 stars and to PRs that touch some spec", "[1] Agarwal et al.", SM,
     "we sweep every pull request that touches a spec in their repositories with at least ten stars"),
    ("98.3% of spec files with a recorded history have a single author", "[1] Agarwal et al.", SM,
     "98.3% of spec files with a recorded history have a single author"),
    ("EPIC offers 40 practices", "[5] Claudino et al.", EP, "EPIC, has 40 practices"),
    ("derived from 114 repositories", "[5] Claudino et al.", EP, "114 open-source SDD repositories"),
    ("most of over 3,000 GitHub projects", "[7] Tan et al.", TA, "over 3,000 GitHub projects and found that most projects"),
    ("the service creates one repository per generated app", "[1] Agarwal et al.", SM, "a generator account emitting one repository per app (13,749)"),
    ("not in sync with other parts of a system", "[6] Aghajani et al.", AG, "not in sync with other parts of a system"),
    ("was correct and complete before a change was introduced", "[6] Aghajani et al.", AG, "was correct and complete before a change was introduced"),
    ("Spec decay affects living specs, mainly when changes alter behavior that other specs describe", "[1] Agarwal et al., Finding 10", SM,
     "Spec decay affects living specs, mainly when changes alter behavior that other specs describe"),
    ("a spec that specifies more implementation detail decays sooner", "[1] Agarwal et al.", SM,
     "a spec that specifies more implementation detail decays sooner"),
    ("merge several edits into one", "[1] Agarwal et al.", SM, "Squashed or rebased commits also merge several edits into one"),
    ("cannot replace all human annotators", "[3] Ahmed et al.", "2408.05534v2.pdf", "we cannot replace all the humans for a sample with the current models"),
]
linhas_c = []
for frase, ref, arq_, trecho in citados:
    t_ = texto(arq_)
    estado = "source not shipped: not checked here" if t_ is None else ("passage found" if trecho in t_ else "PASSAGE NOT FOUND")
    assert estado != "PASSAGE NOT FOUND", (frase, arq_)
    CHECK.append((f"cited: {frase}", frase))
    linhas_c.append((frase, ref, f"{arq_}: {estado}"))
display(pd.DataFrame(linhas_c, columns=["Paper", "Reference", "Source passage"], index=range(1, len(linhas_c) + 1)))

,Paper,Reference,Source passage
1,"21 interviews, and 266 repositories",[1] Agarwal et al.,2608.25202v5.pdf: passage found
2,81.5% of specs had one commit,[1] Agarwal et al.,2608.25202v5.pdf: passage found
3,7 interviewees raised spec decay,"[1] Agarwal et al., Appendix J",2608.25202v5.pdf: passage found
4,"(73,030 in",[1] Agarwal et al.,2608.25202v5.pdf: passage found
5,SpecMine collected PRs only from repositories ...,[1] Agarwal et al.,2608.25202v5.pdf: passage found
6,repositories with at least 10 stars and to PRs...,[1] Agarwal et al.,2608.25202v5.pdf: passage found
7,98.3% of spec files with a recorded history ha...,[1] Agarwal et al.,2608.25202v5.pdf: passage found
8,EPIC offers 40 practices,[5] Claudino et al.,2610.07534.pdf: passage found
9,derived from 114 repositories,[5] Claudino et al.,2610.07534.pdf: passage found
10,"most of over 3,000 GitHub projects",[7] Tan et al.,tan2024_emse_outdated_code_refs.pdf: passage f...


## Check against the paper text

Runs where `paper/main.tex` exists. Two checks: (1) every value above appears in the paper exactly as computed;
(2) every number in the paper (digits, small numbers written out, the verdict words *met*, *refuted*,
*inconclusive*, *negative result*, and quantifiers such as *mostly*, *usually*, *rarely*, *small*, *weak*) lies inside one of
those values, in a declared structural use, or in a declared judgment word whose number is checked beside it. Anything
else fails the notebook, so no number can enter the paper without a source.

In [11]:
ESTRUTURAL = [  # uses that are not data: codebook step numbers, unit descriptions, and the design itself
    "(1) does the diff", "(2) is that behavior", "(3) does a sentence",
    "is one living spec and one PR", "Several linked files in one PR still form one pair", "one-sided",
    "RQ2 rests on one human rater", "for one generator", "give one generator",
    "follow two opposite patterns", "one PR that changed",
    "a script that runs the full pipeline twice",
    "is met when the whole interval lies on the predicted side, refuted when it lies on the other side, and inconclusive otherwise",
    "it assesses artifact quality at one point", "define an outdated document as one", "One RQ2 pair:",
    "experienced at least one such PR", "although the two labeled",
    "did not measure how often it occurs", "are edited more often than Spec Kit specs, reflecting their maintained role",
    "How often are specs edited after creation", "How often does a pull request", "We answer it through two subquestions",
    "H1: OpenSpec living specs are edited more often than Spec Kit specs", "Our data fit three explanations",
    "One RQ2 pair:", "RQ2 rests on one human rater",
    "how often they are revised, and how often code changes leave them outdated",
    "how often does the spec become false or incomplete", "One RQ2 pair on a timeline",
]
INTERPRETACAO = [  # words of judgment; the numbers beside them are checked above
    "Such PRs are common", "a similar number of pairs as decay",
]
tex = f"{ROOT}/paper/main.tex"
if not os.path.exists(tex):
    print("paper/main.tex not found: check skipped")
else:
    def norm(s):
        s = re.sub(r"\\textbf\{([^{}]*)\}", r"\1", s)
        s = re.sub(r"\{\\scriptsize\s*([^{}]*)\}", r"\1", s)
        s = s.replace("\\%", "%").replace("$", "").replace("\\le", "<=").replace("\\ge", ">=").replace("\\kappa", "kappa")
        s = re.sub(r"\\emph\{([^{}]*)\}", r"\1", s)
        s = re.sub(r"\\texttt\{([^{}]*)\}", r"\1", s)
        s = s.replace("\\delta", "delta").replace("~", " ").replace("\\ ", " ").replace("\n", " ")
        return re.sub(r"\s+", " ", s)
    bruto = open(tex, encoding="utf-8").read()
    t = norm(bruto)
    faltam = [(p_, v) for p_, v in CHECK if norm(v) not in t]
    print(f"{len(CHECK) - len(faltam)} of {len(CHECK)} values found in paper/main.tex")
    for p_, v in faltam:
        print("NOT FOUND:", p_, "->", v)
    assert not faltam, "some values differ from the paper text"

    corpo = bruto[bruto.index("\\begin{document}"):]
    corpo = re.sub(r"\\begin\{tikzpicture\}.*?\\end\{tikzpicture\}", " ", corpo, flags=re.S)
    corpo = "\n".join(l for l in corpo.splitlines() if not l.lstrip().startswith("%"))
    corpo = re.sub(r"\\(cite|label|ref|url|includegraphics|bibliography|bibliographystyle|setlength|cmidrule|multicolumn|begin|end)"
                   r"(\([^)]*\))?(\[[^]]*\])?(\{[^{}]*\})*", " ", corpo)
    t2 = norm(corpo)
    coberto = [False] * len(t2)
    print("judgment words, shown with their checked numbers:", INTERPRETACAO)
    for v in [v for _, v in CHECK] + ESTRUTURAL + INTERPRETACAO:
        for m in re.finditer(re.escape(norm(v)), t2):
            coberto[m.start():m.end()] = [True] * (m.end() - m.start())
    TOKEN = re.compile(r"(?<![A-Za-z0-9.])\d[\d,.]*\d(?![A-Za-z0-9])|(?<![A-Za-z0-9.])\d(?![\d.,A-Za-z])"
                       r"|\b(?:zero|one|two|three|four|five|six|seven|eight|nine|ten|eleven|twelve|half|quarters?|twice)\b"
                       r"|\b(?:met|refuted|inconclusive|negative result)\b"
                       r"|\b(?:most|mostly|mainly|usually|rarely|many|common|often|majority|minority|small|weak|similar|older|higher|concentrates)\b", re.I)
    soltos = [(m.group(), t2[max(0, m.start() - 60):m.end() + 40]) for m in TOKEN.finditer(t2) if not all(coberto[m.start():m.end()])]
    for tok, ctx in soltos:
        print(f"NO SOURCE: {tok!r} in ...{ctx}...")
    nome_versao = norm(f"v1.0 ({corte:%B %Y} snapshot)")  # "snapshot" only names the SpecMine version; dates name the collection
    snap_soltos = len(re.findall(r"snapshot", t2, re.I)) - t2.count(nome_versao)
    print(f"uses of 'snapshot' other than the version name ({nome_versao}): {snap_soltos}")
    assert snap_soltos == 0, "use the collection dates, not 'snapshot'"
    print(f"numbers and verdicts in the paper without a source: {len(soltos)}")
    assert not soltos, "every number in the paper needs a source above"


142 of 142 values found in paper/main.tex
judgment words, shown with their checked numbers: ['Such PRs are common', 'a similar number of pairs as decay']
uses of 'snapshot' other than the version name (v1.0 (July 2026 snapshot)): 0
numbers and verdicts in the paper without a source: 0
